# SRQ4 — ViT Baseline Comparison

**Research Question:** How does the best-performing optimised CNN variant compare to a standalone pretrained ViT-B/16 fine-tuned under an identical two-phase protocol — same epoch counts, learning rates, data splits, augmentation, and evaluation metrics — in terms of classification accuracy, prediction calibration, and inference cost?

**Models evaluated (Run 11 in experimental map):**
- `deit_small` — DeiT-Small/16, ~22M params (1.9× ResNet18). Conservative ViT comparison.
- `efficientformer` — EfficientFormer-L1, ~12M params (~1× ResNet18). Parameter-matched comparison.

**Protocol:**
- Identical two-phase protocol to all CNN experiments: Phase 1 trains head only (`param_mode='head'`); Phase 2 fine-tunes everything.
- Hyperparameters loaded from `grid-search-results/optimal_config.csv` (same source as arch-eval).
- 5-fold stratified CV on `X_trainval`; held-out test set untouched until Section 9.
- Best CNN result loaded from `arch-eval-results/arch_eval_results.csv` for direct comparison.

**Key additional metrics (SRQ4-specific):**
- FLOPs — `thop.profile()` on a single 224×224 input
- Inference latency — mean time over 100 forward passes on the test set
- ECE — calibration quality, directly relevant to clinical deployment trust

**Central empirical claim being tested:** CNN variants with attention are competitive with or superior to ViTs under data-scarce, compute-limited conditions. A marginally better ViT costing 5× more is a valid finding supporting the CNN-centric conclusion.

## 1 · Paths & Imports

In [1]:
import sys
from pathlib import Path

ABSOLUTE_PATH = Path().resolve()
PROJECT_ROOT  = ABSOLUTE_PATH.parents[1]
DATA_DIR      = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR   = PROJECT_ROOT / "data" / "experiments" / "vit-comparison-results"
WEIGHTS_DIR   = RESULTS_DIR / "weights"
CURVES_DIR    = RESULTS_DIR / "training-curves"
PLOTS_DIR     = RESULTS_DIR / "plots"

# Upstream experiment directories (read-only)
GRID_SEARCH_RESULTS_DIR   = PROJECT_ROOT / "data" / "experiments" / "grid-search-results"
HEAD_ABLATION_RESULTS_DIR = PROJECT_ROOT / "data" / "experiments" / "head-ablation-results"
ARCH_EVAL_RESULTS_DIR     = PROJECT_ROOT / "data" / "experiments" / "arch-eval-results"

for d in [RESULTS_DIR, WEIGHTS_DIR, CURVES_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Weights dir  : {WEIGHTS_DIR}")
print(f"Curves dir   : {CURVES_DIR}")
print(f"Plots dir    : {PLOTS_DIR}")

Project root : C:\Users\markm\Workspace\ms-machine-learning-diagnosis
Data dir     : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
Results dir  : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results
Weights dir  : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results\weights
Curves dir   : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results\training-curves
Plots dir    : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results\plots


In [2]:
import csv
import time
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.metrics import f1_score

import src.scripts.data      as data
import src.scripts.models    as models
import src.scripts.trainer   as trainer
import src.scripts.utils     as utils
import src.scripts.evaluator as evaluator

utils.set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42 for Python, NumPy, and PyTorch
Device: cpu


## 2 · Data — Outer Split (Identical to All Other Experiments)

Fixed seed 42 outer 80/20 stratified split. SRQ4 operates entirely within `X_trainval` during CV; the held-out test set remains untouched until Section 9.

In [3]:
path, categories = data.get_dataset(str(DATA_DIR))
classes = data.get_classes(path, categories, visualise=False)

image_paths, labels = data.get_paths_and_labels(path, classes)
train_transform, test_transform = data.get_transforms()

X_trainval, y_trainval, X_test, y_test = data.get_trainval_test_split(
    image_paths, labels,
    test_split=0.20,
    SEED=42
)

print(f"\nSRQ4 CV operates on {len(X_trainval)} train+val samples.")
print("Held-out test set is NOT used until Section 9 (final evaluation).")

get_dataset()>>> Dataset already exists in C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
get_dataset()>>> Available categories: ['Control Axial_crop', 'Control Saggital_crop', 'MS Axial_crop', 'MS Saggital_crop']
get_paths_and_labels()>>> Total images: 1652
get_trainval_test_split()>>> Train+Val pool : 1321 (80.0%)
get_trainval_test_split()>>> Held-out test  : 331 (20.0%)
get_trainval_test_split()>>> TrainVal class ratio — MS: 520  Non-MS: 801
get_trainval_test_split()>>> Test     class ratio — MS: 130  Non-MS: 201

SRQ4 CV operates on 1321 train+val samples.
Held-out test set is NOT used until Section 9 (final evaluation).


## 3 · Configuration

Hyperparameters loaded from `grid-search-results/optimal_config.csv`. Identical to arch-eval — this is the 'identical protocol' guarantee that makes SRQ4 a fair comparison.

In [4]:
# ── Load optimal hyperparameters from grid search ────────────────────────────
OPTIMAL_CONFIG_FILE = GRID_SEARCH_RESULTS_DIR / "optimal_config.csv"
if not OPTIMAL_CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"optimal_config.csv not found at {OPTIMAL_CONFIG_FILE}\n"
        "Run the grid search notebook first."
    )

optimal = pd.read_csv(OPTIMAL_CONFIG_FILE).iloc[0]
LR_PHASE1         = float(optimal["lr_phase1"])
WD_PHASE1         = float(optimal["wd_phase1"])
BEST_LR_PHASE2    = float(optimal["lr_phase2"])
BEST_WEIGHT_DECAY = float(optimal["weight_decay"])

print(f"Loaded from : {OPTIMAL_CONFIG_FILE.name}")
print(f"  lr_phase1    = {LR_PHASE1:.0e}")
print(f"  wd_phase1    = {WD_PHASE1}")
print(f"  lr_phase2    = {BEST_LR_PHASE2:.0e}")
print(f"  weight_decay = {BEST_WEIGHT_DECAY:.0e}")

Loaded from : optimal_config.csv
  lr_phase1    = 1e-03
  wd_phase1    = 0.0
  lr_phase2    = 1e-04
  weight_decay = 1e-04


In [5]:
# ── Fixed training parameters (identical to arch-eval) ───────────────────────
N_SPLITS    = 5
BATCH_SIZE  = 32
P1_EPOCHS   = 20
P2_EPOCHS   = 15
P1_PATIENCE = 4
P2_PATIENCE = 3
SEED        = 42

# ViT architectures to evaluate
# NOTE: param_mode for Phase 1 is 'head' (not 'head_and_attention') because
# ViT models have no randomly-initialised CNN attention modules. The transformer
# encoder is pretrained and frozen; only the classification head warms up first.
VIT_ARCHITECTURES = ["deit_small", "efficientformer"]

RESULTS_FILE = RESULTS_DIR / "srq4_results.csv"
CSV_FIELDNAMES = [
    "architecture", "lr_phase1", "wd_phase1", "lr_phase2", "weight_decay",
    "fold", "val_acc", "val_loss", "val_f1", "val_auc", "val_ece",
    "p1_epochs_run", "p2_epochs_run", "timestamp", "error"
]

total_runs = len(VIT_ARCHITECTURES) * N_SPLITS
print(f"{len(VIT_ARCHITECTURES)} architectures × {N_SPLITS} folds = {total_runs} runs")
print(f"  Phase 1: {P1_EPOCHS} epochs max, patience {P1_PATIENCE}  (param_mode='head')")
print(f"  Phase 2: {P2_EPOCHS} epochs max, patience {P2_PATIENCE}  (param_mode='all')")
print(f"Results → {RESULTS_FILE}")
print(f"Weights → {WEIGHTS_DIR}/<architecture>/fold_<n>.pt")

2 architectures × 5 folds = 10 runs
  Phase 1: 20 epochs max, patience 4  (param_mode='head')
  Phase 2: 15 epochs max, patience 3  (param_mode='all')
Results → C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results\srq4_results.csv
Weights → C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\experiments\vit-comparison-results\weights/<architecture>/fold_<n>.pt


## 4 · FLOPs & Latency Profiling Helpers

`thop` measures multiply-add operations for a single 224×224 input. Latency is the mean wall-clock time per batch over 100 forward passes on the test loader. Both metrics are computed on the final trained model (post Phase 2) so they reflect actual inference cost.

In [6]:
try:
    from thop import profile as thop_profile
    THOP_AVAILABLE = True
    print("thop available — FLOPs profiling enabled")
except ImportError:
    THOP_AVAILABLE = False
    print("thop not found. Install with: pip install thop")
    print("FLOPs will be reported as NaN. Latency measurement still works.")


def profile_model(model, device=DEVICE, input_size=(1, 3, 224, 224)):
    """
    Compute FLOPs (GMACs) and parameter count for the model.

    Args:
        model:      Trained PyTorch model in eval mode.
        device:     Inference device.
        input_size: Batch input shape. Default (1, 3, 224, 224).

    Returns:
        flops_g (float): GigaMACs (= GFLOPs / 2 approximately)
        params_m (float): Millions of parameters
    """
    model.eval()
    dummy = torch.randn(*input_size).to(device)

    if THOP_AVAILABLE:
        with torch.no_grad():
            macs, params = thop_profile(model, inputs=(dummy,), verbose=False)
        flops_g  = macs / 1e9
        params_m = params / 1e6
    else:
        flops_g  = float("nan")
        params_m = sum(p.numel() for p in model.parameters()) / 1e6

    return flops_g, params_m


def measure_latency(model, test_loader, device=DEVICE, n_warmup=10, n_measure=100):
    """
    Measure mean per-batch inference latency in milliseconds.

    Runs n_warmup batches first (discarded) to warm up GPU/CPU caches,
    then times n_measure batches and returns the mean.

    Args:
        model:       Trained model in eval mode.
        test_loader: DataLoader providing test batches.
        device:      Inference device.
        n_warmup:    Number of warm-up forward passes (not timed).
        n_measure:   Number of timed forward passes.

    Returns:
        mean_ms (float): Mean batch latency in milliseconds.
        std_ms  (float): Std-dev of batch latency in milliseconds.
    """
    model.eval()
    timings = []
    batch_iter = iter(test_loader)

    with torch.no_grad():
        # Warm-up passes
        for _ in range(n_warmup):
            try:
                imgs, _ = next(batch_iter)
            except StopIteration:
                batch_iter = iter(test_loader)
                imgs, _ = next(batch_iter)
            _ = model(imgs.to(device))

        # Timed passes
        for _ in range(n_measure):
            try:
                imgs, _ = next(batch_iter)
            except StopIteration:
                batch_iter = iter(test_loader)
                imgs, _ = next(batch_iter)
            imgs = imgs.to(device)
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(imgs)
            if device.type == "cuda":
                torch.cuda.synchronize()
            timings.append((time.perf_counter() - t0) * 1000)

    mean_ms = float(np.mean(timings))
    std_ms  = float(np.std(timings))
    return mean_ms, std_ms

thop available — FLOPs profiling enabled


## 5 · CV Helper Functions

In [7]:
def load_completed_runs(results_file):
    """Return set of (architecture, fold) tuples already written without error."""
    completed = set()
    if results_file.exists():
        with open(results_file, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["error"] == "":
                    completed.add((row["architecture"], int(row["fold"])))
    return completed


def append_result(results_file, fieldnames, row_dict):
    """Append one result row to CSV, writing header if the file is new."""
    write_header = not results_file.exists()
    with open(results_file, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row_dict)


def weights_path_for(arch, fold_idx):
    """Return the .pt save path for a given architecture and fold."""
    arch_dir = WEIGHTS_DIR / arch
    arch_dir.mkdir(parents=True, exist_ok=True)
    return arch_dir / f"fold_{fold_idx}.pt"

## 6 · ViT Training Loop (5-Fold CV)

**Protocol note for ViT Phase 1:** `param_mode='head'` freezes the entire pretrained transformer encoder and trains only the randomly-initialised classification head. This mirrors the CNN protocol intent (warm up randomly-initialised components before touching the pretrained backbone) while being correct for ViTs — there are no SE/CBAM modules to train alongside the head.

Results are appended fold-by-fold to `srq4_results.csv`. Safe to interrupt and resume — completed runs are detected automatically.

In [ ]:
completed_runs = load_completed_runs(RESULTS_FILE)
run_number     = len(completed_runs)

for arch in VIT_ARCHITECTURES:
    for fold_idx in range(N_SPLITS):

        run_key = (arch, fold_idx)

        if run_key in completed_runs:
            print(f"SKIP  arch={arch}  fold={fold_idx+1}/{N_SPLITS}")
            continue

        utils.set_seed(SEED)
        run_number += 1
        print(f"\n{'='*65}")
        print(f"  Run {run_number}/{total_runs}  |  arch={arch}  fold={fold_idx+1}/{N_SPLITS}")
        print(f"{'='*65}")

        error_msg = ""
        try:
            train_loader, val_loader = data.get_fold_loaders(
                X_trainval, y_trainval,
                fold_idx=fold_idx,
                train_transform=train_transform,
                test_transform=test_transform,
                n_splits=N_SPLITS,
                batch_size=BATCH_SIZE,
                SEED=SEED
            )

            # ViT head defaults to 'linear' in models.py
            # (transformer already performs non-linear mixing; MLP head is redundant)
            model = models.get_model(architecture=arch)

            run_configs = {
                "phase1": {
                    "num_epochs"  : P1_EPOCHS,
                    "lr"          : LR_PHASE1,
                    "parameters"  : "head",          # ← freeze transformer, train head only
                    "optimiser"   : optim.AdamW,
                    "criterion"   : nn.BCEWithLogitsLoss(),
                    "weight_decay": WD_PHASE1,
                },
                "phase2": {
                    "num_epochs"  : P2_EPOCHS,
                    "lr"          : BEST_LR_PHASE2,
                    "parameters"  : "all",            # ← unfreeze everything
                    "optimiser"   : optim.AdamW,
                    "criterion"   : nn.BCEWithLogitsLoss(),
                    "weight_decay": BEST_WEIGHT_DECAY,
                },
            }

            # ── Phase 1: head warm-up ─────────────────────────────────────────
            losses_p1, accs_p1, val_losses_p1, val_accs_p1 = trainer.train_model(
                model, train_loader, val_loader,
                config_name="phase1",
                train_configs=run_configs,
                early_stopping_patience=P1_PATIENCE
            )
            p1_epochs_run = len(val_accs_p1)

            utils.plot(
                losses_p1, accs_p1,
                config_name=f"phase1_fold{fold_idx}",
                val_losses=val_losses_p1, val_accuracies=val_accs_p1,
                model_name=arch,
                save_dir=str(CURVES_DIR),
                show=False
            )

            # ── Phase 2: full fine-tune ───────────────────────────────────────
            losses_p2, accs_p2, val_losses_p2, val_accs_p2 = trainer.train_model(
                model, train_loader, val_loader,
                config_name="phase2",
                train_configs=run_configs,
                early_stopping_patience=P2_PATIENCE
            )
            p2_epochs_run = len(val_accs_p2)

            utils.plot(
                losses_p2, accs_p2,
                config_name=f"phase2_fold{fold_idx}",
                val_losses=val_losses_p2, val_accuracies=val_accs_p2,
                model_name=arch,
                save_dir=str(CURVES_DIR),
                show=False
            )

            # ── Validation inference: F1, AUC, ECE ───────────────────────────
            model.eval()
            all_preds, all_labels_list, all_probs = [], [], []
            val_running_loss = 0.0
            val_criterion    = nn.BCEWithLogitsLoss()

            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs   = imgs.to(DEVICE)
                    logits = model(imgs).squeeze(1)
                    loss   = val_criterion(logits, lbls.float().to(DEVICE))
                    val_running_loss += loss.item() * imgs.size(0)
                    probs  = torch.sigmoid(logits).cpu().numpy()
                    preds  = (probs >= 0.5).astype(int)
                    all_preds.extend(preds.tolist())
                    all_labels_list.extend(lbls.tolist())
                    all_probs.extend(probs.tolist())

            val_loss = val_running_loss / len(all_labels_list)
            val_acc  = np.mean(np.array(all_preds) == np.array(all_labels_list))
            val_f1   = f1_score(all_labels_list, all_preds, zero_division=0)

            _, _, _, _, val_auc, val_ece, _, _ = evaluator.evaluate_model(
                y_true=all_labels_list, y_pred=all_preds, y_probs=all_probs
            )

            # ── Save weights ──────────────────────────────────────────────────
            save_path = weights_path_for(arch, fold_idx)
            utils.save_weights(model, save_path)

            print(f"\n  → val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
                  f"val_f1={val_f1:.4f}  val_auc={val_auc:.4f}  val_ece={val_ece:.4f}")

        except Exception as e:
            error_msg = str(e)
            print(f"ERROR in {arch} fold {fold_idx+1}: {error_msg}")
            traceback.print_exc()
            val_loss = val_acc = val_f1 = val_auc = val_ece = float("nan")
            p1_epochs_run = p2_epochs_run = 0

        append_result(RESULTS_FILE, CSV_FIELDNAMES, {
            "architecture"  : arch,
            "lr_phase1"     : LR_PHASE1,
            "wd_phase1"     : WD_PHASE1,
            "lr_phase2"     : BEST_LR_PHASE2,
            "weight_decay"  : BEST_WEIGHT_DECAY,
            "fold"          : fold_idx,
            "val_acc"       : round(val_acc,  4),
            "val_loss"      : round(val_loss, 4),
            "val_f1"        : round(val_f1,   4),
            "val_auc"       : round(val_auc,  4),
            "val_ece"       : round(val_ece,  4),
            "p1_epochs_run" : p1_epochs_run,
            "p2_epochs_run" : p2_epochs_run,
            "timestamp"     : datetime.now().isoformat(timespec="seconds"),
            "error"         : error_msg,
        })

print(f"\n{'='*65}")
print("SRQ4 CV COMPLETE")
print(f"Results → {RESULTS_FILE}")

Random seed set to 42 for Python, NumPy, and PyTorch

  Run 1/10  |  arch=deit_small  fold=1/5
get_fold_loaders()>>> Fold 1/5 — Train: 1056,  Val: 265
get_model()>>> architecture='deit_small'  head='mlp'
[phase1] Epoch 1/20 - Train Loss: 0.4978 - Train Acc: 0.7472 - Val Loss: 0.4141 - Val Acc: 0.8189
[phase1] Epoch 2/20 - Train Loss: 0.3739 - Train Acc: 0.8381 - Val Loss: 0.3723 - Val Acc: 0.8302
[phase1] Epoch 3/20 - Train Loss: 0.3280 - Train Acc: 0.8608 - Val Loss: 0.3395 - Val Acc: 0.8604
[phase1] Epoch 4/20 - Train Loss: 0.2946 - Train Acc: 0.8920 - Val Loss: 0.3906 - Val Acc: 0.7849
[phase1] Epoch 5/20 - Train Loss: 0.2977 - Train Acc: 0.8788 - Val Loss: 0.3135 - Val Acc: 0.8679
[phase1] Epoch 6/20 - Train Loss: 0.2753 - Train Acc: 0.8854 - Val Loss: 0.2864 - Val Acc: 0.8830
[phase1] Epoch 7/20 - Train Loss: 0.2609 - Train Acc: 0.8958 - Val Loss: 0.2945 - Val Acc: 0.8868
[phase1] Epoch 8/20 - Train Loss: 0.2473 - Train Acc: 0.8987 - Val Loss: 0.3833 - Val Acc: 0.7774
[phase1] Epo

## 7 · CV Results Summary — ViT Models

In [ ]:
df_vit = pd.read_csv(RESULTS_FILE)
df_vit = df_vit[df_vit["error"] == ""]

df_vit_summary = df_vit.groupby("architecture").agg(
    mean_val_f1   = ("val_f1",   "mean"),
    std_val_f1    = ("val_f1",   "std"),
    mean_val_auc  = ("val_auc",  "mean"),
    std_val_auc   = ("val_auc",  "std"),
    mean_val_ece  = ("val_ece",  "mean"),
    std_val_ece   = ("val_ece",  "std"),
    mean_val_acc  = ("val_acc",  "mean"),
    std_val_acc   = ("val_acc",  "std"),
    mean_val_loss = ("val_loss", "mean"),
    n_folds       = ("fold",     "count"),
).round(4)

df_vit_summary = df_vit_summary.sort_values("mean_val_f1", ascending=False)

print("ViT model CV summary (sorted by mean val F1):")
print(df_vit_summary.to_string())

## 8 · Load Best CNN from Arch-Eval

Load the best-performing CNN architecture from `arch-eval-results/arch_eval_results.csv`. Selection criterion mirrors arch-eval: highest mean val F1; ties broken by mean val acc then lowest mean val loss.

In [ ]:
ARCH_EVAL_CSV = ARCH_EVAL_RESULTS_DIR / "arch_eval_results.csv"
if not ARCH_EVAL_CSV.exists():
    raise FileNotFoundError(
        f"arch_eval_results.csv not found at {ARCH_EVAL_CSV}\n"
        "Run the arch-eval notebook first."
    )

df_arch = pd.read_csv(ARCH_EVAL_CSV)
df_arch = df_arch[df_arch["error"] == ""]

df_cnn_summary = df_arch.groupby("architecture").agg(
    mean_val_f1   = ("val_f1",   "mean"),
    std_val_f1    = ("val_f1",   "std"),
    mean_val_auc  = ("val_auc",  "mean"),
    std_val_auc   = ("val_auc",  "std"),
    mean_val_ece  = ("val_ece",  "mean"),
    std_val_ece   = ("val_ece",  "std"),
    mean_val_acc  = ("val_acc",  "mean"),
    n_folds       = ("fold",     "count"),
).round(4)

df_cnn_summary = df_cnn_summary.sort_values(
    ["mean_val_f1", "mean_val_acc", "mean_val_ece"],
    ascending=[False, False, True]
)

BEST_CNN_ARCH = df_cnn_summary.index[0]
best_cnn_row  = df_cnn_summary.loc[BEST_CNN_ARCH]

print(f"Best CNN architecture : {BEST_CNN_ARCH}")
print(f"  mean_val_f1  = {best_cnn_row['mean_val_f1']:.4f} ± {best_cnn_row['std_val_f1']:.4f}")
print(f"  mean_val_auc = {best_cnn_row['mean_val_auc']:.4f} ± {best_cnn_row['std_val_auc']:.4f}")
print(f"  mean_val_ece = {best_cnn_row['mean_val_ece']:.4f} ± {best_cnn_row['std_val_ece']:.4f}")
print(f"  mean_val_acc = {best_cnn_row['mean_val_acc']:.4f} ± {best_cnn_row['std_val_acc']:.4f}")
print(f"\nFull CNN CV summary:")
print(df_cnn_summary[["mean_val_f1", "std_val_f1", "mean_val_auc", "mean_val_ece", "mean_val_acc"]].to_string())

## 9 · FLOPs & Latency — All Models

FLOPs and latency are measured on the best-fold model (fold with highest val F1) for each architecture. This is the model that will be used for the final test set evaluation, so these figures reflect actual inference cost at deployment.

**Interpreting the results:** A marginally better ViT costing 5× more in FLOPs is a valid finding that *supports* the CNN-centric conclusion under clinical resource constraints.

In [ ]:
# ── Load test loader (needed for latency measurement) ────────────────────────
test_loader = data.get_test_loader(X_test, y_test, test_transform, batch_size=BATCH_SIZE)

efficiency_results = {}

# ── Measure ViT models ────────────────────────────────────────────────────────
for arch in VIT_ARCHITECTURES:
    arch_rows   = df_vit[df_vit["architecture"] == arch]
    best_fold   = int(arch_rows.loc[arch_rows["val_f1"].idxmax(), "fold"])
    weights_path = weights_path_for(arch, best_fold)

    model = models.get_model(architecture=arch)
    model = utils.load_weights(model, weights_path, device=DEVICE)

    flops_g, params_m = profile_model(model, device=DEVICE)
    lat_mean, lat_std = measure_latency(model, test_loader, device=DEVICE)

    efficiency_results[arch] = {
        "flops_g" : round(flops_g,   3),
        "params_m": round(params_m,  2),
        "lat_ms"  : round(lat_mean,  3),
        "lat_std" : round(lat_std,   3),
    }
    print(f"{arch}: {flops_g:.2f} GMACs  |  {params_m:.1f}M params  |  {lat_mean:.2f} ± {lat_std:.2f} ms/batch")

# ── Measure best CNN model ────────────────────────────────────────────────────
cnn_arch_rows  = df_arch[df_arch["architecture"] == BEST_CNN_ARCH]
cnn_best_fold  = int(cnn_arch_rows.loc[cnn_arch_rows["val_f1"].idxmax(), "fold"])
cnn_weights    = ARCH_EVAL_RESULTS_DIR / "weights" / BEST_CNN_ARCH / f"fold_{cnn_best_fold}.pt"

# Load winning head from head ablation (same as arch-eval)
OPTIMAL_HEAD_FILE = HEAD_ABLATION_RESULTS_DIR / "optimal_head.csv"
WINNING_HEAD = pd.read_csv(OPTIMAL_HEAD_FILE).iloc[0]["head"] if OPTIMAL_HEAD_FILE.exists() else "linear"

cnn_model = models.get_model(architecture=BEST_CNN_ARCH, head=WINNING_HEAD)
cnn_model = utils.load_weights(cnn_model, cnn_weights, device=DEVICE)

cnn_flops_g, cnn_params_m = profile_model(cnn_model, device=DEVICE)
cnn_lat_mean, cnn_lat_std = measure_latency(cnn_model, test_loader, device=DEVICE)

efficiency_results[BEST_CNN_ARCH] = {
    "flops_g" : round(cnn_flops_g,  3),
    "params_m": round(cnn_params_m, 2),
    "lat_ms"  : round(cnn_lat_mean, 3),
    "lat_std" : round(cnn_lat_std,  3),
}
print(f"{BEST_CNN_ARCH} (best CNN): {cnn_flops_g:.2f} GMACs  |  {cnn_params_m:.1f}M params  |  {cnn_lat_mean:.2f} ± {cnn_lat_std:.2f} ms/batch")

## 10 · Final Test Set Evaluation

Run ONCE on the held-out test set after all model selection and CV decisions are made. Each model uses the best-fold weights (fold with highest val F1 from CV above).

In [ ]:
test_results = {}

# ── Evaluate ViT models ───────────────────────────────────────────────────────
for arch in VIT_ARCHITECTURES:
    print(f"\n{'─'*50}")
    print(f"  Test evaluation: {arch}")
    print(f"{'─'*50}")

    arch_rows  = df_vit[df_vit["architecture"] == arch]
    best_fold  = int(arch_rows.loc[arch_rows["val_f1"].idxmax(), "fold"])
    weights_p  = weights_path_for(arch, best_fold)

    model = models.get_model(architecture=arch)
    model = utils.load_weights(model, weights_p, device=DEVICE)

    acc, prec, rec, f1, auc, ece, conf, report = evaluator.evaluate_model(
        model=model, test_loader=test_loader, device=DEVICE
    )

    eff = efficiency_results.get(arch, {})
    test_results[arch] = {
        "test_acc" : acc,  "test_prec": prec, "test_rec": rec,
        "test_f1"  : f1,   "test_auc" : auc,  "test_ece": ece,
        "flops_g"  : eff.get("flops_g",  float("nan")),
        "params_m" : eff.get("params_m", float("nan")),
        "lat_ms"   : eff.get("lat_ms",   float("nan")),
        "conf"     : conf,
    }

# ── Evaluate best CNN ─────────────────────────────────────────────────────────
print(f"\n{'─'*50}")
print(f"  Test evaluation: {BEST_CNN_ARCH} (best CNN)")
print(f"{'─'*50}")

cnn_model = models.get_model(architecture=BEST_CNN_ARCH, head=WINNING_HEAD)
cnn_model = utils.load_weights(cnn_model, cnn_weights, device=DEVICE)

acc, prec, rec, f1, auc, ece, conf, report = evaluator.evaluate_model(
    model=cnn_model, test_loader=test_loader, device=DEVICE
)

eff = efficiency_results.get(BEST_CNN_ARCH, {})
test_results[BEST_CNN_ARCH] = {
    "test_acc" : acc,  "test_prec": prec, "test_rec": rec,
    "test_f1"  : f1,   "test_auc" : auc,  "test_ece": ece,
    "flops_g"  : eff.get("flops_g",  float("nan")),
    "params_m" : eff.get("params_m", float("nan")),
    "lat_ms"   : eff.get("lat_ms",   float("nan")),
    "conf"     : conf,
}

# ── Build comparison DataFrame ────────────────────────────────────────────────
df_test = pd.DataFrame({
    k: {m: v for m, v in v.items() if m != "conf"}
    for k, v in test_results.items()
}).T.astype(float).round(4)

print("\n" + "="*60)
print("SRQ4 — Final Test Set Comparison")
print("="*60)
print(df_test[["test_f1", "test_auc", "test_ece", "test_acc",
               "flops_g", "params_m", "lat_ms"]].to_string())

In [ ]:
# ── Save test results to CSV ──────────────────────────────────────────────────
TEST_RESULTS_FILE = RESULTS_DIR / "srq4_test_results.csv"
df_test.to_csv(TEST_RESULTS_FILE)
print(f"Test results saved → {TEST_RESULTS_FILE}")

## 11 · SRQ4 Visualisations

Four panels:
1. **F1 bar chart** — performance comparison, CNN vs ViTs
2. **AUC-ROC vs ECE** — discriminability vs calibration
3. **FLOPs vs F1 scatter** — efficiency–performance trade-off (the SRQ4 centrepiece)
4. **Latency bar chart** — wall-clock inference cost

In [ ]:
if not df_test.empty:

    # ── Colour scheme: CNN = amber, ViTs = blue shades ────────────────────────
    COLOUR_MAP = {
        BEST_CNN_ARCH : "#DD8452",   # amber  — CNN
        "deit_small"  : "#4C72B0",   # blue   — DeiT
        "efficientformer": "#55A868",# green  — EfficientFormer
    }
    colours = [COLOUR_MAP.get(a, "#888888") for a in df_test.index]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("SRQ4 — CNN vs ViT Baseline Comparison", fontsize=15, fontweight="bold")

    plot_order = df_test.sort_values("test_f1", ascending=True)
    bar_colours = [COLOUR_MAP.get(a, "#888888") for a in plot_order.index]

    # ── Panel 1: F1 bar chart ─────────────────────────────────────────────────
    ax = axes[0, 0]
    y_pos = np.arange(len(plot_order))
    ax.barh(y_pos, plot_order["test_f1"], color=bar_colours,
            alpha=0.85, height=0.55, edgecolor="white", linewidth=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_order.index, fontsize=10)
    ax.set_xlabel("Test F1", fontsize=11)
    ax.set_title("SRQ4 — Test F1", fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    f1_vals = plot_order["test_f1"].values
    pad = max((f1_vals.max() - f1_vals.min()) * 0.3, 0.02)
    ax.set_xlim(max(0, f1_vals.min() - pad), min(1, f1_vals.max() + pad))
    for y, v in zip(y_pos, f1_vals):
        ax.text(v + 0.003, y, f"{v:.4f}", va="center", fontsize=9)

    # ── Panel 2: AUC-ROC vs ECE grouped bars ─────────────────────────────────
    ax2 = axes[0, 1]
    x = np.arange(len(df_test))
    w = 0.35
    arch_labels = list(df_test.index)
    ax2.bar(x - w/2, df_test["test_auc"], w, label="AUC-ROC",
            color="#4C72B0", alpha=0.82)
    ax2.bar(x + w/2, df_test["test_ece"], w, label="ECE (lower=better)",
            color="#DD8452", alpha=0.82)
    ax2.set_xticks(x)
    ax2.set_xticklabels(arch_labels, rotation=20, ha="right", fontsize=9)
    ax2.set_ylabel("Score", fontsize=11)
    ax2.set_title("AUC-ROC & ECE (Calibration)", fontsize=11, fontweight="bold")
    ax2.legend(fontsize=9)
    ax2.spines[["top", "right"]].set_visible(False)

    # ── Panel 3: FLOPs vs F1 scatter (efficiency–performance trade-off) ───────
    ax3 = axes[1, 0]
    for arch_name, row in df_test.iterrows():
        c = COLOUR_MAP.get(arch_name, "#888888")
        ax3.scatter(row["flops_g"], row["test_f1"], color=c, s=120,
                    zorder=3, label=arch_name)
        ax3.annotate(arch_name, (row["flops_g"], row["test_f1"]),
                     textcoords="offset points", xytext=(6, 4), fontsize=8)
    ax3.set_xlabel("FLOPs (GMACs)", fontsize=11)
    ax3.set_ylabel("Test F1", fontsize=11)
    ax3.set_title("Efficiency–Performance Trade-off", fontsize=11, fontweight="bold")
    ax3.legend(fontsize=8)
    ax3.grid(True, linestyle="--", alpha=0.4)
    ax3.spines[["top", "right"]].set_visible(False)

    # ── Panel 4: Inference latency bar chart ──────────────────────────────────
    ax4 = axes[1, 1]
    lat_order = df_test.sort_values("lat_ms", ascending=True)
    lat_colours = [COLOUR_MAP.get(a, "#888888") for a in lat_order.index]
    y4 = np.arange(len(lat_order))
    ax4.barh(y4, lat_order["lat_ms"], color=lat_colours,
             alpha=0.85, height=0.55, edgecolor="white", linewidth=0.8)
    ax4.set_yticks(y4)
    ax4.set_yticklabels(lat_order.index, fontsize=10)
    ax4.set_xlabel("Mean Latency (ms / batch)", fontsize=11)
    ax4.set_title("Inference Latency", fontsize=11, fontweight="bold")
    ax4.spines[["top", "right"]].set_visible(False)
    for y, v in zip(y4, lat_order["lat_ms"].values):
        ax4.text(v + 0.1, y, f"{v:.1f} ms", va="center", fontsize=9)

    plt.tight_layout()
    utils.save_fig(fig, PLOTS_DIR, "srq4_comparison")
    plt.show()
    plt.close(fig)

## 12 · Summary Table

Formatted table for direct transcription into the dissertation. Columns ordered to match the SRQ4 narrative: performance first (F1, AUC, Acc), then calibration (ECE), then cost (FLOPs, params, latency).

In [ ]:
if not df_test.empty:
    display_cols = [
        "test_f1", "test_auc", "test_acc", "test_prec", "test_rec",
        "test_ece", "flops_g", "params_m", "lat_ms"
    ]
    rename_map = {
        "test_f1"  : "F1",
        "test_auc" : "AUC-ROC",
        "test_acc" : "Accuracy",
        "test_prec": "Precision",
        "test_rec" : "Recall",
        "test_ece" : "ECE ↓",
        "flops_g"  : "GMACs",
        "params_m" : "Params (M)",
        "lat_ms"   : "Latency (ms)",
    }

    df_display = df_test[display_cols].rename(columns=rename_map)

    # Highlight best value in each column
    print("SRQ4 Final Comparison Table:")
    print("(↑ = higher better | ECE, GMACs, Params, Latency = ↓ lower better)")
    print()
    print(df_display.to_string())

    # ── FLOPs relative to CNN baseline ────────────────────────────────────────
    if BEST_CNN_ARCH in df_test.index and not np.isnan(df_test.loc[BEST_CNN_ARCH, "flops_g"]):
        cnn_flops = df_test.loc[BEST_CNN_ARCH, "flops_g"]
        print(f"\nFLOPs relative to best CNN ({BEST_CNN_ARCH}):")
        for arch_name in df_test.index:
            ratio = df_test.loc[arch_name, "flops_g"] / cnn_flops
            delta_f1 = df_test.loc[arch_name, "test_f1"] - df_test.loc[BEST_CNN_ARCH, "test_f1"]
            print(f"  {arch_name:<35} {ratio:.2f}×   ΔF1 = {delta_f1:+.4f}")